# Notebook 2: Real-Time Pipeline Demonstration

This notebook demonstrates a single end-to-end run of the airport optimization pipeline. It shows how to:

1. Initialize the main `AirportOptimizationEngine`.
2. Fetch and process real-time and external data (congestion, weather, passenger forecasts).
3. Define a passenger profile.
4. Request a route recommendation for a specific scenario.
5. Display the results and explanation.

This provides an interactive way to test the core logic of the system.

In [ ]:
import sys
import os
from datetime import datetime, timedelta

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.db import get_engine, FeatureStore
from src.pipeline.engine import AirportOptimizationEngine
from src.models.personalization import PassengerProfile
from src.utils.helpers import print_recommendations, parse_time_hhmm

### 1. Initialize the Optimization Engine

This sets up all the necessary components: data ingestors, feature builders, predictors, and the optimization agent.

In [ ]:
try:
    engine = get_engine()
    feature_store = FeatureStore(engine)
    app_engine = AirportOptimizationEngine(engine, feature_store)
    print("✅ AirportOptimizationEngine initialized successfully.")
except Exception as e:
    print(f"❌ Engine initialization failed: {e}")

### 2. Define a Passenger and Scenario

Here we define the context for the recommendation. We'll simulate a passenger arriving by taxi with a specific flight time.

In [ ]:
# Define the passenger profile. Leaving fields as None will trigger Bayesian imputation.
passenger = PassengerProfile(
    age_group='adult',
    mobility='normal',
    bags=2,
    companions=1
)

# Define the scenario
origin_type = 'taxi' # Corresponds to Q3
origin_location = '3층_8번_정차구역' # A specific taxi drop-off point

# Set the required time at the gate (e.g., boarding time)
# Let's set it to 90 minutes from now.
required_gate_time = datetime.now() + timedelta(minutes=90)

print(f"Passenger Profile: {passenger}")
print(f"Scenario: Arriving by {origin_type} at '{origin_location}'")
print(f"Required at gate by: {required_gate_time.strftime('%Y-%m-%d %H:%M')}")

### 3. Get Route Recommendation

This is the core call. The engine will perform the full data pipeline (fetch, process, predict, optimize) to generate the recommendation.

In [ ]:
print("Requesting route recommendation... (This may take a moment as it fetches live data)")

try:
    recommendation_result = app_engine.recommend_route(
        profile=passenger,
        origin_type=origin_type,
        origin_location=origin_location,
        required_gate_time=required_gate_time
    )
    print("\nRecommendation received.")
except Exception as e:
    print(f"❌ Failed to get recommendation: {e}")
    recommendation_result = None

### 4. Display Results

In [ ]:
if recommendation_result:
    print_recommendations("Live Route Recommendation", recommendation_result)
else:
    print("No result to display.")